# About

This markdown recapitulates the detection of cell cycle-related genes and the subclustering of G1-G2 cells to detect genes that might be useful for grouping/classifying cells as G1 or G2.

# 1. Neoblast Score

## Importing modules and settings

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
from matplotlib.backends.backend_pdf import PdfPages
import openpyxl

In [ ]:
import seaborn as sns

In [ ]:
import anndata as ad

In [ ]:
import scanpy.external as sce

In [ ]:
import random

General settings of Scanpy

In [ ]:
sc.settings.verbosity = 3 
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')

In [ ]:
#umap_cmap = sns.blend_palette(['xkcd:light grey', 'xkcd:indigo'], as_cmap = True)
plt.rcParams['figure.figsize'] = [10,10]
grayurple = ["lightgrey","#a100d2","#4c1f89"]
umap_cmap = sns.blend_palette(grayurple, as_cmap = True)

## Declaring the input and output files

In [ ]:
# Reading the adata file
adata = sc.read_h5ad('./../../analysis_250528_f/Smed_L78-L47_20250523_Annotated.h5ad') 

Selecting one clustering layer from leiden_names

In [ ]:
# Select the Leiden resolution that you used for clustering
clusteringlayer = 'annotated_names'

In [ ]:
# Input the Excel annotation file filled after deciding how to annotate all clusters
# this file is provided in 'matrix processing'
df = pd.read_excel('./../../../data/cell_annotation/cluster_annotation_250523.xlsx', index_col = 'ID')

In [ ]:
df = df.sort_index()

In [ ]:
df['cluster names'].value_counts()

In [ ]:
with rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color='annotated_names', legend_fontoutline = 5,
               size = 30,
               #palette = list(dfs['Colour'].unique()),
               frameon=False)

In [ ]:
with rc_context({'figure.figsize': (25, 5)}):
    sns.barplot(adata.obs['annotated_names'].value_counts(), palette = list(df['colour hex'].unique()))
    plt.xticks(rotation=90)
    plt.axvline(9.5, color = 'black', linewidth = 1)
    plt.axvline(37.5, color = 'black', linewidth = 1)
    plt.axvline(43.5, color = 'black', linewidth = 1)
    plt.axvline(53.5, color = 'black', linewidth = 1)
    plt.axvline(61.5, color = 'black', linewidth = 1)
    plt.axvline(64.5, color = 'black', linewidth = 1)
    plt.axvline(78.5, color = 'black', linewidth = 1)
    plt.axvline(81.5, color = 'black', linewidth = 1)
    plt.axvline(82.5, color = 'black', linewidth = 1)
    plt.axvline(83.5, color = 'black', linewidth = 1)
    plt.axvline(85.5, color = 'black', linewidth = 1)

In [ ]:
# retrieve the neoblast score
neoblast_score = pd.DataFrame.from_dict(adata.uns['neoblast_score_leiden_2.5'], orient='index', columns=['neoblast_score'])
neoblast_score.index.name = 'leiden_2.5'

In [ ]:
# Bar plot of the neoblast score
data = neoblast_score.sort_values('neoblast_score', ascending = False)

li_colours = [] # order the colours according to the order of the sorted clusters
for i in data.index.astype(int):
    li_colours.append(df[df.index == i]['colour hex'][i])

with plt.rc_context({'figure.figsize': (20, 5)}):
    sns.barplot(x=data.index.tolist(), y='neoblast_score', data= data, palette = li_colours )
    plt.xticks(rotation=90)
    plt.title('neoblast score per cluster')

## Neoblast score with the final annotation

In [ ]:
score_cluster = adata.uns['neoblast_score_annotated_names']

In [ ]:
score = (pd.DataFrame.from_dict(score_cluster, orient="index", columns=["neoblast_score"])
      .sort_values(by="neoblast_score", ascending=True))

In [ ]:
score

In [ ]:
df = pd.DataFrame({
    "cluster names": adata.obs["annotated_names"].cat.categories,
    "colour hex": adata.uns["annotated_names_colors"]})

In [ ]:
df

In [ ]:
# Bar plot of the neoblast score
data = score.sort_values('neoblast_score', ascending = False)

li_colours = [] # order the colours according to the order of the sorted clusters
for i in data.index:
    li_colours.append(df[df['cluster names'] == i]['colour hex'].tolist()[0])

with plt.rc_context({'figure.figsize': (20, 5)}):
    sns.barplot(x=data.index.tolist(), y='neoblast_score', data= data, palette = li_colours )
    plt.xticks(rotation=90)
    plt.title('neoblast score per cluster')

# 3. Subsetting the adata to keep G1/G2 neoblast/progenitor cells

We will now subset the adata object to keep only the cells that belong to our dataset of interest (G1/G2) and that belong to our clusters of interest (neoblasts or progenitors).

First we create a filter to subset by experiment. We are only interested in the `FACS` experiment.

In [ ]:
f1 = adata.obs['Experiment'] == 'FACS'

Then we will subset by cell cluster, only for neoblasts and progenitors.
To make it more specific, we will first define the clusters of interests as those above a threshold value for neoblast score.

In [ ]:
f2 = adata.obs['broad_names'] == 'neoblasts'

Then we will create a filter that interrogates *every individual cell in the dataset*, to fish the cells qualifying as high neoblast score no matter whether they are from neo+progenitor clusters or not.

In [ ]:
f3 = adata.obs['neoblast_score'] > 0.2589 

These two filters are boolean evaluation. We can group them using `&` and doing some order of parentheses, to subset the adata.

In [ ]:
#f_cyc = ((f1) & (f3))
f_cyc = (f1) & ((f2) | (f3))

In [ ]:
cells_of_interest = adata[f_cyc,].obs.index

In [ ]:
len(cells_of_interest)

## load raw unprocessed matrix with only the right cell barcode IDs

make sure that we have raw integers and preserve them somewhere, likely using layers such as `adata_neo.layers['counts'] = ... `
all of these tips and info can be found at https://discourse.scverse.org/t/how-could-adata-raw-x-contain-non-integer-values/3708/2

In [ ]:
adata_u = sc.read_h5ad('./../../analysis_250528_f/Smed_L78-L47_20250523_unprocessed.h5ad')

In [ ]:
sum(adata_u.obs.index == adata.obs.index) == len(adata.obs.index)

Here we make an AnnData from scratch for the cells with high neoblast score. We call this `adata_neo`.

In [ ]:
X_ = adata_u.X[f_cyc.values, :]
adata_neo = ad.AnnData(X_)
# check stuff:
# type(X_)
# adata_neo.X.max()

As seen below, this "new" adata has fewer cells.

In [ ]:
adata_neo

We add the `adata.obs` and `adata.var`

In [ ]:
adata_neo.obs = adata.obs[f_cyc]
adata_neo.obs.index = adata.obs.index[f_cyc]
adata_neo.var = adata.raw.var

In [ ]:
adata_neo

## Writing for DGE

We will write down this matrix to perform countsplit and DGE (see R markdown).

In [ ]:
# Extract the obs and vars
adata_neo.obs.to_csv('./cells_metadata_for_DGE.csv') # obs
adata_neo.var.to_csv('./all_genes_metadata_for_DGE.csv') # var, all

In [ ]:
# Get the raw counts matrix
raw_counts = adata_neo.X

In [ ]:
import gzip
from scipy.io import mmwrite

# Define the output file path
output_file = "./raw_for_DGE.mtx.gz"

# Save the raw counts matrix as .mtx.gz
with gzip.open(output_file, 'wb') as f:
    mmwrite(f, raw_counts)

We will also save the adata_neo object:

In [ ]:
adata_neo.write('./adata_neoblastscore_unprocessed.h5ad', compression='gzip')